# 01 — NIEA river water-quality exploration

This notebook investigates the unchanged **NIEA River Water Quality Monitoring 1990–2024 — All Parameters** CSV. It does not clean, remove or overwrite observations. See `docs/dataset.md` for provenance and the data dictionary.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET_PATH = PROJECT_ROOT / 'data' / 'raw' / 'niea_river_water_quality_1990_2024.csv'
OFFICIAL_SOURCE = 'https://www.data.gov.uk/dataset/0840ab33-18ae-4670-8523-ac43105f5902/river-water-quality-monitoring-1990-to-2024-all-parameters'

if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f'Missing selected dataset: {DATASET_PATH}\n'
        f'Download the unchanged official CSV from: {OFFICIAL_SOURCE}'
    )

# Blank and whitespace-only source fields represent absent entries in this audit.
data = pd.read_csv(
    DATASET_PATH, encoding='utf-8-sig', na_values=[' ', ''], low_memory=False
)
data['Date_parsed'] = pd.to_datetime(data['Date'], format='%Y/%m/%d', errors='coerce')
print(f'Loaded unchanged raw file: {DATASET_PATH.name}')

## Provenance reminder

The file comes from the Northern Ireland Environment Agency Water Science and Evidence Unit and is published through OpenDataNI under the UK Open Government Licence. The source warns that typos, sampling error or contamination may produce extreme results. Raw data cannot provide official water status without appropriate processing.

## Shape, columns, types and first rows

In [ ]:
print(f'Shape: {data.shape[0]:,} rows × {data.shape[1] - 1} original columns')
schema = pd.DataFrame({
    'column': data.drop(columns='Date_parsed').columns,
    'dtype': data.drop(columns='Date_parsed').dtypes.astype(str).values,
})
display(schema)
display(data.drop(columns='Date_parsed').head())

## Missing values

Missingness is reported rather than repaired. Detection-limit qualifiers such as `<` are not missing and are reviewed separately.

In [ ]:
missing = pd.DataFrame({
    'missing_count': data.drop(columns='Date_parsed').isna().sum(),
    'missing_percent': data.drop(columns='Date_parsed').isna().mean().mul(100).round(3),
}).sort_values(['missing_percent', 'missing_count'], ascending=False)
display(missing)

## Duplicate and redundant records

In [ ]:
original_columns = data.drop(columns='Date_parsed')
print('Exact duplicate rows:', int(original_columns.duplicated().sum()))
print('Duplicate station/date/time keys:', int(data.duplicated(['StationCode', 'Date', 'Time']).sum()))
print('X duplicates Easting for every row:', bool(data['X'].equals(data['Easting'])))
print('Y duplicates Northing for every row:', bool(data['Y'].equals(data['Northing'])))
constant_columns = [
    column for column in original_columns.columns
    if original_columns[column].nunique(dropna=True) <= 1
]
print('Constant or all-missing columns:', constant_columns)

## Numeric and categorical variables

In [ ]:
analysis_data = data.drop(columns='Date_parsed')
numeric_columns = analysis_data.select_dtypes(include='number').columns.tolist()
categorical_columns = [column for column in analysis_data.columns if column not in numeric_columns]
print(f'Numeric columns ({len(numeric_columns)}):', numeric_columns)
print(f'Categorical/other columns ({len(categorical_columns)}):', categorical_columns)

## Chemistry summary and distributions

The plotted variables are central to the research question and comparatively well populated. Metals remain in the numeric summary; their greater missingness makes combined plots less readable.

In [ ]:
measurement_columns = [
    'ALK_mg_l_', 'BOD_mg_l_', 'COND_US_CM_', 'D_Cu_µg_l_', 'D_FE_µg_l_',
    'DO_mg_l_', 'NH4_N_mg_l_', 'NO2_N_mg_l_', 'NO3_N_mg_l_',
    'P_SOL__mg_l_', 'PH_PHUNITS_', 'SS_mg_l_', 'ZNSOL_µg_l_'
]
important_measurements = [
    'BOD_mg_l_', 'COND_US_CM_', 'DO_mg_l_', 'NH4_N_mg_l_',
    'NO2_N_mg_l_', 'NO3_N_mg_l_', 'P_SOL__mg_l_', 'PH_PHUNITS_'
]
display(data[measurement_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T)
data[important_measurements].hist(bins=40, figsize=(14, 10))
plt.suptitle('Selected chemistry distributions — raw values retained')
plt.tight_layout()
plt.show()

## Detection and quantification qualifiers

A `<` flag indicates a censored result rather than a fully observed value. No substitution rule is applied in this exploration.

In [ ]:
qualifier_columns = [column for column in data.columns if column.startswith('Q_')]
qualifier_summary = pd.DataFrame({
    '< count': [(data[column] == '<').sum() for column in qualifier_columns],
    '> count': [(data[column] == '>').sum() for column in qualifier_columns],
    'blank count': [data[column].isna().sum() for column in qualifier_columns],
}, index=qualifier_columns)
display(qualifier_summary)

## Target review

There is no classification target. Dissolved oxygen is inspected only as a **provisional continuous regression outcome**; it is not converted into risk classes.

In [ ]:
TARGET_COLUMN = 'DO_mg_l_'
target = data[TARGET_COLUMN]
print(f'Non-missing target observations: {target.notna().sum():,}')
print(f'Missing target observations: {target.isna().sum():,} ({target.isna().mean() * 100:.3f}%)')
display(target.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).to_frame())
target.dropna().plot(kind='hist', bins=50, figsize=(9, 4), title='Raw dissolved-oxygen distribution')
plt.xlabel('Dissolved oxygen (mg/L)')
plt.show()

## Time coverage

Record counts show monitoring effort, not water quality. Unequal counts must not be interpreted as environmental trends.

In [ ]:
print('Earliest date:', data['Date_parsed'].min().date())
print('Latest date:', data['Date_parsed'].max().date())
print('Unparseable dates:', int(data['Date_parsed'].isna().sum()))
annual_counts = data['Date_parsed'].dt.year.value_counts().sort_index()
annual_counts.plot(kind='bar', figsize=(14, 4), title='Sampling records by year')
plt.ylabel('Records')
plt.xlabel('Year')
plt.tight_layout()
plt.show()

## Location coverage

In [ ]:
print('Unique station codes:', data['StationCode'].nunique(dropna=True))
print('Unique location names:', data['Location'].nunique(dropna=True))
print('Unique non-missing primary basins:', data['PrimaryBasin'].nunique(dropna=True))
basin_counts = data['PrimaryBasin'].value_counts().head(20).sort_values()
basin_counts.plot(kind='barh', figsize=(9, 7), title='Twenty basins with most sampling records')
plt.xlabel('Records')
plt.tight_layout()
plt.show()

## Correlation analysis

Pearson correlations use pairwise available raw values. They may be affected by censoring, outliers, missingness, station differences and time. Correlation does not establish causation.

In [ ]:
correlation = data[important_measurements].corr()
fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(correlation, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(correlation.columns)), correlation.columns, rotation=90)
ax.set_yticks(range(len(correlation.index)), correlation.index)
fig.colorbar(image, ax=ax, label='Pearson correlation')
ax.set_title('Pairwise raw-value correlations')
plt.tight_layout()
plt.show()
display(correlation.round(3))

## Suspicious and impossible-looking values

The checks below flag negative results, pH outside the conventional 0–14 mathematical scale, non-finite values, and observations outside 1.5 × IQR. IQR flags are statistical, not scientific invalidity rules. Nothing is removed.

In [ ]:
review_rows = []
for column in measurement_columns:
    values = pd.to_numeric(data[column], errors='coerce')
    observed = values.dropna()
    q1, q3 = observed.quantile([0.25, 0.75])
    iqr = q3 - q1
    outside_iqr = (observed < q1 - 1.5 * iqr) | (observed > q3 + 1.5 * iqr)
    review_rows.append({
        'column': column,
        'observed': int(observed.size),
        'minimum': observed.min(),
        '99th_percentile': observed.quantile(0.99),
        'maximum': observed.max(),
        'negative_count': int((observed < 0).sum()),
        'outside_1.5_iqr_review_only': int(outside_iqr.sum()),
        'non_finite_count': int((~np.isfinite(observed)).sum()),
    })
display(pd.DataFrame(review_rows).set_index('column'))
outside_ph_scale = data['PH_PHUNITS_'].notna() & ~data['PH_PHUNITS_'].between(0, 14)
print('pH observations outside 0–14:', int(outside_ph_scale.sum()))
print('Rows with all 13 chemistry results missing:', int(data[measurement_columns].isna().all(axis=1).sum()))

## Preliminary observations

- The file has useful scale and unusually long river-monitoring coverage.
- No risk or safety target exists; classification is not supported.
- Dissolved oxygen is a plausible continuous target for a later regression baseline, with important limitations documented in `docs/dataset.md`.
- Missingness differs strongly by measurement; dissolved iron and zinc are especially sparse.
- `<` qualifier flags are common and require a censoring decision before modelling.
- Extreme values require station/date/source review, not automatic outlier deletion.
- Repeated stations and dates make a simple random row split scientifically weak for claims about new sites or future periods.
- The raw data have not been cleaned, filtered or used to train a model in this phase.